# Partitioning

## What is Partitioning?

Partitioning is a technique used to **divide a large table into smaller, logical sections** (partitions) based on the values of a particular column (the *partition column*).

## Example: Orders Table

Suppose an `orders` table contains several years of data. Partitioning the table by `order_date` organizes the data into **separate partitions based on the date or month**:

```sql
CREATE TABLE orders (
  order_id   BIGINT,
  customer_id BIGINT,
  amount     DECIMAL(10,2),
  order_date DATE
)
USING DELTA
PARTITIONED BY (order_date);
```

## Partition Pruning

Instead of scanning every partition, Databricks can **identify the relevant partition and skip the others**. This technique is called **partition pruning**.

```sql
SELECT * FROM orders WHERE order_date = '2024-05-01';
-- Only the partition(s) for that date/month are scanned
```

## Main Advantages

- **Reduced data scanning** — queries only read the relevant partitions.
- **Faster queries** when filtering on the **partition column** (e.g. `order_date`).

## Caveats / Disadvantages

- **Too many partitions** (over-partitioning) can result in a **small number of files per partition**, which introduces **additional overhead** (metadata management, small-file problems, slow writes).
- **Changing the partitioning strategy** of an existing table is **not straightforward** — it usually requires rewriting/recreating the table, not just changing a configuration.

> 💡 **Tip:** Choose partition columns with low-to-moderate cardinality (e.g. date, region) and expect queries to filter on them frequently.

## Interview Answer: What is Partitioning?

**Question: What is partitioning in Databricks/Spark, and why do we use it?**

"Partitioning is a technique where a large table is physically divided into smaller, manageable chunks called partitions, based on the values of one or more columns — for example, partitioning an `orders` table by `order_date`, so each date or month gets its own set of files on storage.

**Why it helps:**

1. **Partition pruning** — when a query filters on the partition column, like `WHERE order_date = '2024-05-01'`, Spark reads only the relevant partitions' metadata and skips the rest, dramatically reducing I/O and improving query speed.
2. **Parallelism** — partitions are the unit of parallelism in Spark; each partition can be processed by a separate core/task.

**When NOT to use it / pitfalls:**

- **Over-partitioning** — too many partitions (e.g., partitioning by a high-cardinality column like `user_id`) creates too many small files, adding metadata overhead and slowing down both writes and file listing.
- **Data skew** — if one partition key dominates (e.g., 90% of orders on one date), that partition becomes a hotspot and processing becomes uneven.

**Best practice:** partition on low-to-moderate cardinality columns that are frequently used in filter/join conditions — commonly dates, country, or region. In modern Delta tables, **liquid clustering** is often preferred over static partitioning because it handles data skew and high-cardinality columns better."

**Key one-liner:** *"Partitioning splits a table into smaller physical chunks by a key column so queries can prune unnecessary data and run in parallel — but too many partitions leads to the small-file problem, so choose low-cardinality, frequently-filtered columns."*

In [0]:
# --- Partitioning demo: create an orders table partitioned by order_date ---

from datetime import date
from decimal import Decimal

orders = [
    (1, 101, Decimal("250.00"), date(2024, 5, 1)),
    (2, 102, Decimal("80.50"),  date(2024, 5, 1)),
    (3, 103, Decimal("420.10"), date(2024, 6, 15)),
    (4, 104, Decimal("99.99"),  date(2024, 6, 15)),
    (5, 105, Decimal("310.00"), date(2024, 7, 20)),
]

orders_df = spark.createDataFrame(orders, schema="order_id LONG, customer_id LONG, amount DECIMAL(10,2), order_date DATE")

# Write as a Delta table PARTITIONED BY order_date
# -> creates one sub-directory per date, e.g. .../order_date=2024-05-01/
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .saveAsTable("demo_partitioned_orders")

print("Table written. Displaying it:")
display(spark.table("demo_partitioned_orders"))

In [0]:
# --- Practical example of the interview answer concepts ---
# Demonstrates: 1) partitioning by order_date, 2) partition pruning,
# 3) the over-partitioning / small-file pitfall

from datetime import date
from pyspark.sql.functions import expr

# ------------------------------------------------------------------
# 1) Create a reasonably sized orders table, partitioned by order_date
#    (~3 years of daily data -> ~1000 partitions, a sane number)
# ------------------------------------------------------------------
orders_df = (
    spark.range(0, 300_000)
    .withColumn("order_date", expr("date_add(date '2022-01-01', CAST(id % 1000 AS INT))"))
    .withColumn("customer_id", (expr("id % 500") + 1).cast("BIGINT"))
    .withColumn("amount", (expr("id % 500") * 1.75 + 5).cast("DECIMAL(10,2)"))
    .select("id", "customer_id", "amount", "order_date")
)

(orders_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("order_date")
    .saveAsTable("demo_orders_partitioned"))

num_partitions = spark.sql("SHOW PARTITIONS demo_orders_partitioned").count()
print(f"Table created with {num_partitions} partitions (one per day).")

# ------------------------------------------------------------------
# 2) Partition pruning: filter on the partition column
#    -> the query plan should show 'PartitionFilters: [order_date ...]',
#       meaning Spark lists/reads ONLY the matching partition directory
# ------------------------------------------------------------------
pruned = spark.sql(
    "SELECT * FROM demo_orders_partitioned WHERE order_date = '2023-06-15'"
)
pruned.explain("formatted")  # look for PartitionFilters in the plan
print(f"Rows scanned: {pruned.count()}  (only 1 of {num_partitions} partitions read)")

# ------------------------------------------------------------------
# 3) The pitfall: over-partitioning by a HIGH-cardinality column
#    (customer_id) -> ~500 tiny files, one per customer, small-file problem
# ------------------------------------------------------------------
(orders_df.limit(50_000)
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("customer_id")
    .saveAsTable("demo_orders_badly_partitioned"))

over_parts = spark.sql("SHOW PARTITIONS demo_orders_badly_partitioned").count()
print(f"Badly partitioned table: {over_parts} partitions for only 50,000 rows.")
print("Each partition holds ~100 rows -> many tiny files, high metadata overhead,")
print("slow writes and slow file listing. Modern alternative: liquid clustering:")
print("  CREATE TABLE ... CLUSTER BY (order_date)  -- handles skew/cardinality better.")

# Clean up demos
for t in ("demo_orders_partitioned", "demo_orders_badly_partitioned"):
    spark.sql(f"DROP TABLE IF EXISTS {t}")